# Lab 1.3 — Handling Imbalanced Datasets: Rare Failures vs. Normal Operations

**Module 1 | AI/ML Intermediate Workshop | Nutanix Engineering**

---

## The Problem: Class Imbalance in Infrastructure Monitoring

Imagine your Nutanix cluster generates **10,000 telemetry events per day**. On a well-managed cluster, maybe **200–500 of those events indicate an anomaly** — a precursor to disk failure, memory pressure, or a CPU bottleneck that will cascade into an outage.

That's a **2–5% failure rate**. Everything else is normal.

Now here's the trap:

> _"If 98% of servers are healthy, a model that **always predicts 'healthy'** is 98% accurate — but it completely useless. It will never catch a single failure."_

This is the **class imbalance problem**. Standard ML algorithms optimize for overall accuracy, which means they are biased toward the majority class (normal operations). Rare but critical events — the anomalies, failures, and hardware faults you actually care about — get ignored.

In production infrastructure monitoring, **a missed failure is catastrophic**. A false alarm is annoying. The cost asymmetry is enormous.

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Explain** why accuracy is a misleading metric for imbalanced datasets
2. **Apply** four techniques to address class imbalance:
   - Class weights (built-in, zero data modification)
   - Random oversampling (duplicate minority samples)
   - SMOTE (generate synthetic minority samples)
   - Random undersampling (reduce majority samples)
3. **Select** the right evaluation metrics: Precision, Recall, F1, PR-AUC
4. **Combine** techniques for production-grade anomaly detection
5. **Visualize** the effect of each technique on your training data

---

## Prerequisites

- Completed Lab 1.1 (Feature Engineering) and Lab 1.2 (Feature Selection)
- Familiarity with scikit-learn pipelines and classification metrics

**Estimated time:** 90 minutes

---

> **Instructor Note:** Before starting, ask the class: _"Has anyone ever deployed a model that had great test accuracy but failed in production?"_ This is almost always due to class imbalance or distribution shift. Ground the discussion in real Nutanix operational experience — AHV host monitoring, CVM health checks, storage controller alerts.

## 📦 Requirements & Troubleshooting

### Required Packages

| Package | Install Name |
|---------|-------------|
| pandas | `pandas` |
| numpy | `numpy` |
| matplotlib | `matplotlib` |
| seaborn | `seaborn` |
| scikit-learn | `scikit-learn` |
| imbalanced-learn | `imbalanced-learn` |

**Install all at once:**
```bash
pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn
```

---

### ⚠️ Common Errors & Fixes

**`ModuleNotFoundError: No module named '...'`**
> Package is missing from the active Python environment.
> Fix: Run the pip install command above in a terminal, then **restart the kernel**.

**`CalledProcessError` — `--break-system-packages` / exit status 2**
> You are using a virtual environment (e.g. `myenv`) where that flag is not supported, or your pip version is old.
> Fix: Open a terminal, activate your venv (`source myenv/bin/activate`), then run `pip install <package>` without that flag.

**`Failed building wheel for <package>` / C extension errors**
> The package does not support your Python version (most common on Python 3.14).
> Fix: Switch the kernel to **Python 3.13**. Click the kernel name in the VS Code top-right corner → *Select Another Kernel* → *Python 3.13*. Then re-run.

**Packages install with no error but `ModuleNotFoundError` still appears**
> You installed into a different Python than the one the notebook is using.
> Fix: Check the kernel shown in the top-right of VS Code. Open a terminal, activate that environment, and install packages there.

**`PermissionError` or `[Errno 13]` when installing**
> Trying to install into a read-only system Python.
> Fix: Use a virtual environment — `python -m venv myenv && source myenv/bin/activate` — then install.


## Setup: Import Libraries

We use **scikit-learn** for models and evaluation, and **imbalanced-learn** (`imblearn`) — a companion library built specifically for handling class imbalance. It integrates cleanly with scikit-learn pipelines.

If `imbalanced-learn` is not installed, run: `pip install imbalanced-learn`

In [ ]:
# Core data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# scikit-learn: data splitting, metrics
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score
)

# scikit-learn: models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

# imbalanced-learn: resampling strategies
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler, NearMiss

# Utilities
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Reproducibility seed — important for consistent results across runs
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully.")
print("Ready to explore imbalanced datasets!")

---

## Step 1: Generate an Imbalanced Infrastructure Dataset

We will simulate **server telemetry data** from a Nutanix cluster. Each row represents a snapshot of a single node's health metrics over a 5-minute window.

### Features

| Feature | Description | Units |
|---|---|---|
| `cpu_percent` | CPU utilization | % |
| `memory_percent` | Memory utilization | % |
| `disk_io_mbps` | Disk I/O throughput | MB/s |
| `network_mbps` | Network throughput | MB/s |
| `response_time_ms` | API response latency | ms |
| `error_rate` | Error rate | errors/min |
| `temperature_celsius` | CPU temperature | °C |
| `uptime_hours` | Node uptime since last reboot | hours |

### Target
- `is_anomaly = 1` → The node is in a failure precursor state (rare, ~5%)
- `is_anomaly = 0` → Normal operations (common, ~95%)

> **Instructor Note:** Walk through the data generation logic. Point out that anomalies are defined by **combinations** of high values — high CPU *and* high memory *and* high response time. A single high-CPU reading might just be a VM migration. This reflects real-world complexity where no single feature tells the whole story.

In [ ]:
def generate_nutanix_telemetry(n_samples=2000, anomaly_rate=0.05, random_state=42):
    """
    Generates synthetic Nutanix cluster node telemetry data.
    
    Normal nodes operate within healthy ranges.
    Anomalous nodes show elevated CPU, memory, response time, and error rates —
    characteristic of pre-failure states like memory leaks, I/O saturation,
    or cascading VM failures.
    """
    rng = np.random.default_rng(random_state)
    n_anomalies = int(n_samples * anomaly_rate)
    n_normal = n_samples - n_anomalies

    # ----------------------------------------------------------------
    # Normal node telemetry — healthy operating ranges
    # ----------------------------------------------------------------
    normal_data = {
        'cpu_percent':        rng.uniform(20, 70, n_normal),       # moderate CPU load
        'memory_percent':     rng.uniform(30, 70, n_normal),       # moderate memory pressure
        'disk_io_mbps':       rng.uniform(10, 300, n_normal),      # typical I/O range
        'network_mbps':       rng.uniform(5, 500, n_normal),       # typical network traffic
        'response_time_ms':   rng.uniform(1, 50, n_normal),        # low latency
        'error_rate':         rng.uniform(0, 0.5, n_normal),       # near-zero errors
        'temperature_celsius':rng.uniform(35, 65, n_normal),       # normal operating temp
        'uptime_hours':       rng.uniform(1, 8760, n_normal),      # up to 1 year uptime
        'is_anomaly':         np.zeros(n_normal, dtype=int)
    }

    # ----------------------------------------------------------------
    # Anomalous node telemetry — pre-failure signatures
    # High CPU + high memory + degraded response times + elevated errors
    # ----------------------------------------------------------------
    anomaly_data = {
        'cpu_percent':        rng.uniform(88, 100, n_anomalies),   # near-saturation CPU
        'memory_percent':     rng.uniform(83, 98, n_anomalies),    # memory pressure / near OOM
        'disk_io_mbps':       rng.uniform(400, 800, n_anomalies),  # I/O saturation
        'network_mbps':       rng.uniform(600, 950, n_anomalies),  # network congestion
        'response_time_ms':   rng.uniform(200, 2000, n_anomalies), # severe latency degradation
        'error_rate':         rng.uniform(3, 20, n_anomalies),     # elevated error rate
        'temperature_celsius':rng.uniform(70, 95, n_anomalies),    # thermal throttling risk
        'uptime_hours':       rng.uniform(1, 8760, n_anomalies),   # failures can happen anytime
        'is_anomaly':         np.ones(n_anomalies, dtype=int)
    }

    # Combine and shuffle
    df_normal  = pd.DataFrame(normal_data)
    df_anomaly = pd.DataFrame(anomaly_data)
    df = pd.concat([df_normal, df_anomaly], ignore_index=True)
    df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)  # shuffle rows

    # Add small Gaussian noise to make the dataset more realistic
    numeric_cols = [c for c in df.columns if c != 'is_anomaly']
    noise = rng.normal(0, 0.5, size=(len(df), len(numeric_cols)))
    df[numeric_cols] = df[numeric_cols].values + noise

    # Clip values to realistic bounds
    df['cpu_percent']        = df['cpu_percent'].clip(0, 100)
    df['memory_percent']     = df['memory_percent'].clip(0, 100)
    df['disk_io_mbps']       = df['disk_io_mbps'].clip(0, 1000)
    df['network_mbps']       = df['network_mbps'].clip(0, 1000)
    df['response_time_ms']   = df['response_time_ms'].clip(0.1, 5000)
    df['error_rate']         = df['error_rate'].clip(0, 30)
    df['temperature_celsius']= df['temperature_celsius'].clip(20, 100)
    df['uptime_hours']       = df['uptime_hours'].clip(0, 8760)

    return df


# Generate the dataset
df = generate_nutanix_telemetry(n_samples=2000, anomaly_rate=0.05)

print(f"Dataset shape: {df.shape}")
print(f"\nColumn types:")
print(df.dtypes)
print(f"\nFirst 5 rows:")
df.head()

### Examine the Class Distribution

Let's quantify the imbalance and visualize it. A visual representation makes the severity immediately obvious.

In [ ]:
# Count class distribution
class_counts = Counter(df['is_anomaly'])
total = len(df)

print("=" * 50)
print("CLASS DISTRIBUTION")
print("=" * 50)
for label, count in sorted(class_counts.items()):
    label_name = "Anomaly (Failure)" if label == 1 else "Normal Operation"
    pct = count / total * 100
    bar = "█" * int(pct / 2)
    print(f"  Class {label} ({label_name:22s}): {count:5d} samples ({pct:5.1f}%) {bar}")

print(f"\n  Imbalance ratio: {class_counts[0] / class_counts[1]:.1f}:1 (normal:anomaly)")

# ----------------------------------------------------------------
# Visualization: class distribution bar chart
# ----------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart — absolute counts
labels = ['Normal\n(is_anomaly=0)', 'Anomaly\n(is_anomaly=1)']
counts = [class_counts[0], class_counts[1]]
colors = ['#2196F3', '#F44336']  # Blue for normal, Red for anomaly

bars = axes[0].bar(labels, counts, color=colors, edgecolor='black', linewidth=0.8, width=0.5)
axes[0].set_title('Class Distribution (Absolute Counts)', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Number of Samples')
axes[0].set_ylim(0, max(counts) * 1.2)

# Add count labels on bars
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
                 f'{count}', ha='center', va='bottom', fontweight='bold', fontsize=12)

# Pie chart — proportions
wedge_colors = ['#2196F3', '#F44336']
explode = (0, 0.1)  # Slightly pull out the anomaly slice to emphasize it
axes[1].pie(
    counts,
    labels=[f'Normal\n({counts[0]/total*100:.1f}%)', f'Anomaly\n({counts[1]/total*100:.1f}%)'],
    colors=wedge_colors,
    explode=explode,
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 11}
)
axes[1].set_title('Class Distribution (Proportional)', fontweight='bold', fontsize=13)

plt.suptitle('Nutanix Cluster Telemetry — Class Imbalance Visualization',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nKey Insight: The anomaly slice is barely visible in the pie chart.")
print("This is what imbalanced data looks like in production infrastructure monitoring.")

### Explore Feature Distributions

Let's see how the features differ between normal and anomalous states. This builds intuition for what "failure" looks like in the data.

In [ ]:
# Feature columns (exclude target)
feature_cols = ['cpu_percent', 'memory_percent', 'disk_io_mbps', 'network_mbps',
                'response_time_ms', 'error_rate', 'temperature_celsius', 'uptime_hours']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

normal_df  = df[df['is_anomaly'] == 0]
anomaly_df = df[df['is_anomaly'] == 1]

for i, col in enumerate(feature_cols):
    axes[i].hist(normal_df[col], bins=40, alpha=0.6, color='#2196F3',
                 label='Normal', density=True, edgecolor='none')
    axes[i].hist(anomaly_df[col], bins=40, alpha=0.75, color='#F44336',
                 label='Anomaly', density=True, edgecolor='none')
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].legend(fontsize=9)

plt.suptitle('Feature Distributions: Normal vs. Anomalous Nodes',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary statistics comparison
print("\nMean feature values by class:")
summary = df.groupby('is_anomaly')[feature_cols].mean().round(2)
summary.index = ['Normal (0)', 'Anomaly (1)']
print(summary.T.to_string())

### Prepare Train/Test Split

We use **stratified splitting** — this ensures both train and test sets preserve the original class ratio. Without stratification, random chance could put all anomalies in one split.

> **Instructor Note:** Emphasize **stratified splitting** as a baseline best practice with imbalanced data. Always use `stratify=y` in `train_test_split`. Also explain that **we only resample the training data** — the test set must always reflect the real-world distribution. Never resample the test set.

In [ ]:
X = df[feature_cols].copy()
y = df['is_anomaly'].copy()

# Stratified split: preserves class proportions in both train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y          # <-- critical for imbalanced data
)

# Scale features — important for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit ONLY on training data
X_test_scaled  = scaler.transform(X_test)         # apply same scale to test

print("Train/Test Split (stratified):")
print(f"  Training samples : {len(X_train):5d} | Anomaly rate: {y_train.mean():.3f}")
print(f"  Test samples     : {len(X_test):5d} | Anomaly rate: {y_test.mean():.3f}")
print(f"\nTraining class counts : {Counter(y_train)}")
print(f"Test class counts     : {Counter(y_test)}")
print("\nBoth splits maintain the ~5% anomaly rate. Stratification works.")

---

## Step 2: The Naive Approach — Why Accuracy Is Lying to You

Let's train a standard Logistic Regression classifier with **no adjustments for class imbalance**. This is the mistake most engineers make when they first build a failure detection model.

We will show:
1. High overall accuracy — looks great!
2. Terrible recall for the anomaly class — it's completely useless in practice

> **Instructor Note:** This is the "gotcha" moment. Let students guess the result before showing the confusion matrix. Ask: _"If this model predicts all zeros, what's its accuracy?"_ Then reveal the classification report. The contrast between the impressive accuracy number and the catastrophic recall should be visceral.

In [ ]:
# ----------------------------------------------------------------
# Naive Logistic Regression — no imbalance handling
# ----------------------------------------------------------------
lr_naive = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
    # Note: class_weight is NOT set — default is None (equal weights)
)
lr_naive.fit(X_train_scaled, y_train)
y_pred_naive = lr_naive.predict(X_test_scaled)

# ----------------------------------------------------------------
# Evaluate
# ----------------------------------------------------------------
print("=" * 65)
print("NAIVE LOGISTIC REGRESSION — No Imbalance Handling")
print("=" * 65)

# Overall accuracy
accuracy = (y_pred_naive == y_test).mean()
print(f"\nOverall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print("  ^ This LOOKS impressive. But let's dig deeper...\n")

# Dummy baseline: always predict 'normal'
dummy_accuracy = (y_test == 0).mean()
print(f"Baseline Accuracy (always predict 'normal'): {dummy_accuracy:.4f} ({dummy_accuracy*100:.2f}%)")
print("  ^ A model that DOES NOTHING achieves this accuracy.")
print("  ^ If our model barely beats this, it's not learning anything useful.\n")

# Detailed classification report
print("-" * 65)
print("Classification Report:")
print("-" * 65)
print(classification_report(y_test, y_pred_naive,
                             target_names=['Normal (0)', 'Anomaly (1)']))

# Confusion matrix
cm_naive = confusion_matrix(y_test, y_pred_naive)
print("Confusion Matrix:")
print(cm_naive)
tn, fp, fn, tp = cm_naive.ravel()
print(f"\n  True Negatives  (correctly identified normal): {tn}")
print(f"  False Positives (normal flagged as anomaly)  : {fp}")
print(f"  False Negatives (MISSED anomalies)           : {fn}  <-- THIS IS THE PROBLEM")
print(f"  True Positives  (correctly caught anomalies) : {tp}")

In [ ]:
# Visualize the confusion matrix
fig, ax = plt.subplots(figsize=(7, 6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_naive,
    display_labels=['Normal', 'Anomaly']
)
disp.plot(ax=ax, colorbar=True, cmap='Blues')

ax.set_title(
    'Naive Model — Confusion Matrix\n'
    '(High accuracy, but misses most failures)',
    fontweight='bold', fontsize=12
)

# Annotate the critical cell
ax.text(0, 1, '← MISSED\n   FAILURES',
        color='red', fontweight='bold', fontsize=10,
        ha='center', va='center', transform=ax.transData)

plt.tight_layout()
plt.show()

recall_naive = recall_score(y_test, y_pred_naive)
print(f"\nAnomaly Recall: {recall_naive:.3f}")
print("\nThis model catches very few anomalies.")
print("In an infrastructure monitoring context, this means:\n")
print("  - Most server failures will be MISSED")
print("  - Ops teams will not be alerted before outages")
print("  - The model provides FALSE confidence in system health")
print("\nThis model is WORSE than useless — it's actively misleading.")

---

## Step 3: Technique 1 — Class Weights

The simplest fix: tell the algorithm that **missing an anomaly is more costly than a false alarm**.

Most scikit-learn classifiers accept a `class_weight` parameter. Setting `class_weight='balanced'` automatically computes weights inversely proportional to class frequencies:

$$w_j = \frac{n_{\text{samples}}}{n_{\text{classes}} \times n_{\text{samples}_j}}$$

If anomalies are 5% of data, they get **19× more weight** in the loss function. The algorithm stops ignoring them.

**Advantages:**
- No data modification required
- Works with LogisticRegression, SVM, RandomForest, GradientBoosting, and most sklearn estimators
- Zero computational overhead
- Works with your original feature distributions

> **Instructor Note:** This is the **first thing you should try** in any imbalanced classification problem. It's the fastest, has no downside, and often works well enough. Demo this live — change one parameter, get dramatically better results. That contrast is powerful.

In [ ]:
# Calculate what 'balanced' class weights actually are
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
weight_dict = dict(zip(classes, weights))

print("Computed class weights for 'balanced' mode:")
print(f"  Normal (class 0)  : weight = {weight_dict[0]:.4f}")
print(f"  Anomaly (class 1) : weight = {weight_dict[1]:.4f}")
print(f"  Ratio             : {weight_dict[1]/weight_dict[0]:.1f}x")
print(f"\n  Each anomaly sample counts {weight_dict[1]/weight_dict[0]:.1f}x")
print(f"  as much as a normal sample during training.")

# ----------------------------------------------------------------
# Logistic Regression with class_weight='balanced'
# ----------------------------------------------------------------
lr_weighted = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',   # <-- the only change
    random_state=RANDOM_STATE
)
lr_weighted.fit(X_train_scaled, y_train)
y_pred_weighted = lr_weighted.predict(X_test_scaled)

print("\n" + "=" * 65)
print("LOGISTIC REGRESSION with class_weight='balanced'")
print("=" * 65)
print(classification_report(y_test, y_pred_weighted,
                             target_names=['Normal (0)', 'Anomaly (1)']))

In [ ]:
# Side-by-side confusion matrix comparison
cm_weighted = confusion_matrix(y_test, y_pred_weighted)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Naive model
ConfusionMatrixDisplay(
    confusion_matrix=cm_naive,
    display_labels=['Normal', 'Anomaly']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(
    f'Naive (No Weighting)\nAnomaly Recall: {recall_score(y_test, y_pred_naive):.2f}',
    fontweight='bold', color='darkred', fontsize=12
)

# Weighted model
ConfusionMatrixDisplay(
    confusion_matrix=cm_weighted,
    display_labels=['Normal', 'Anomaly']
).plot(ax=axes[1], colorbar=False, cmap='Greens')
axes[1].set_title(
    f'Class Weights (balanced)\nAnomaly Recall: {recall_score(y_test, y_pred_weighted):.2f}',
    fontweight='bold', color='darkgreen', fontsize=12
)

plt.suptitle('Confusion Matrix Comparison: Before vs. After Class Weighting',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Compare key metrics
print("\nKey Metric Comparison:")
print(f"{'Metric':<30} {'Naive':>10} {'Weighted':>10}")
print("-" * 52)
print(f"{'Overall Accuracy':<30} {(y_pred_naive==y_test).mean():>10.3f} {(y_pred_weighted==y_test).mean():>10.3f}")
print(f"{'Anomaly Recall':<30} {recall_score(y_test, y_pred_naive):>10.3f} {recall_score(y_test, y_pred_weighted):>10.3f}")
print(f"{'Anomaly Precision':<30} {precision_score(y_test, y_pred_naive, zero_division=0):>10.3f} {precision_score(y_test, y_pred_weighted):>10.3f}")
print(f"{'Anomaly F1 Score':<30} {f1_score(y_test, y_pred_naive):>10.3f} {f1_score(y_test, y_pred_weighted):>10.3f}")

print("\nObservation: Accuracy dropped slightly (we now flag some false alarms),")
print("but anomaly Recall improved substantially. That's the right tradeoff for ops.")

---

## Step 4: Technique 2 — Random Oversampling

**Idea:** Duplicate existing anomaly samples randomly until the classes are balanced. The algorithm then sees an equal number of normal and anomalous examples during training.

**How it works:**
- From your 100 anomaly samples, randomly sample (with replacement) to create 1900 anomaly samples
- Your training set is now balanced: 1900 normal + 1900 anomaly

**Caution:** You are literally **copying the same samples multiple times**. If the model memorizes these duplicates, it will overfit to the specific anomaly patterns in your training data and fail on new, unseen failure modes.

> **Instructor Note:** Ask the class: _"What's the risk of just copying the same 100 failure examples 19 times?"_ Guide them to overfitting. The model may learn the exact CPU/memory values from your training anomalies, rather than generalizing the *pattern* of what constitutes a failure.

In [ ]:
# ----------------------------------------------------------------
# Random Oversampling — duplicate minority class samples
# ----------------------------------------------------------------
ros = RandomOverSampler(random_state=RANDOM_STATE)
X_train_ros, y_train_ros = ros.fit_resample(X_train_scaled, y_train)

print("=" * 55)
print("RANDOM OVERSAMPLING")
print("=" * 55)
print(f"\nBefore resampling: {Counter(y_train)}")
print(f"After resampling : {Counter(y_train_ros)}")
print(f"\nTraining set grew from {len(y_train)} to {len(y_train_ros)} samples")
print(f"Added {len(y_train_ros) - len(y_train)} duplicate anomaly samples")

# Visualize new distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

before_counts = [Counter(y_train)[0], Counter(y_train)[1]]
after_counts  = [Counter(y_train_ros)[0], Counter(y_train_ros)[1]]

x = np.arange(2)
width = 0.35
bars1 = axes[0].bar(x - width/2, before_counts, width, label='Before', color=['#2196F3', '#F44336'], alpha=0.6)
bars2 = axes[0].bar(x + width/2, after_counts,  width, label='After',  color=['#2196F3', '#F44336'], alpha=1.0)
axes[0].set_title('Class Counts: Before vs After Random Oversampling', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(['Normal', 'Anomaly'])
axes[0].set_ylabel('Sample Count')
axes[0].legend()

for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=9)

# Feature distribution: did oversampling change the distributions?
# (It shouldn't — it just repeats existing points)
axes[1].hist(X_train_scaled[y_train == 1, 0], bins=20, alpha=0.6, color='orange',
             label='Original Anomalies', density=True)
axes[1].hist(X_train_ros[y_train_ros == 1, 0], bins=20, alpha=0.4, color='red',
             label='After Oversampling', density=True)
axes[1].set_title('CPU % Distribution: Anomalies (Scaled)\nOversampling preserves original distribution', fontweight='bold')
axes[1].set_xlabel('CPU % (scaled)')
axes[1].legend()

plt.tight_layout()
plt.show()

# Train and evaluate
lr_ros = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_ros.fit(X_train_ros, y_train_ros)
y_pred_ros = lr_ros.predict(X_test_scaled)

print("\nClassification Report (Random Oversampling):")
print(classification_report(y_test, y_pred_ros, target_names=['Normal (0)', 'Anomaly (1)']))

---

## Step 5: Technique 3 — SMOTE (Synthetic Minority Over-sampling Technique)

SMOTE is a smarter form of oversampling. Instead of duplicating existing samples, it **generates brand new synthetic anomaly examples** by interpolating between existing ones.

### How SMOTE Works (Intuitively)

1. Take a real anomaly sample (e.g., a node with CPU=95%, Memory=92%, Latency=800ms)
2. Find its **k nearest neighbors** among other anomaly samples
3. Pick one neighbor at random
4. Create a new synthetic sample **somewhere on the line between** the two anomalies

Think of it like this: if you have two real server failure signatures, SMOTE creates a plausible "third" failure that's a blend of the two. The model learns to recognize the *region* of failure space, not just the exact points.

```
Real anomaly A  ●────────────●  Real anomaly B
                    ★  ← SMOTE synthetic sample
```

**This addresses the overfitting concern** with random oversampling because the model sees varied examples, not duplicates.

> **Instructor Note:** Draw this on the whiteboard. The key insight is that SMOTE creates samples in the **minority class neighborhood** in feature space. It doesn't create random noise — it stays close to real examples. However, SMOTE can generate unrealistic samples if the minority class is highly sparse or in a complex feature space. Always validate with domain experts.

In [ ]:
# ----------------------------------------------------------------
# SMOTE — Synthetic Minority Over-sampling
# ----------------------------------------------------------------
smote = SMOTE(
    k_neighbors=5,         # use 5 nearest neighbors to generate each synthetic sample
    random_state=RANDOM_STATE
)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("=" * 55)
print("SMOTE — Synthetic Minority Over-sampling")
print("=" * 55)
print(f"\nBefore SMOTE: {Counter(y_train)}")
print(f"After SMOTE : {Counter(y_train_smote)}")
print(f"\nGenerated {Counter(y_train_smote)[1] - Counter(y_train)[1]} NEW synthetic anomaly samples")
print(f"None of these are duplicates of existing data — they are interpolated.")

# Verify SMOTE creates truly new samples (not copies)
original_anomalies = X_train_scaled[y_train == 1]
all_anomalies_after_smote = X_train_smote[y_train_smote == 1]
synthetic_only = all_anomalies_after_smote[len(original_anomalies):]

print(f"\nVerification:")
print(f"  Original anomaly samples         : {len(original_anomalies)}")
print(f"  Total anomaly samples after SMOTE: {len(all_anomalies_after_smote)}")
print(f"  Synthetic (new) samples          : {len(synthetic_only)}")

# Check that synthetic samples are NOT exact copies
is_copy = np.any(np.all(synthetic_only[:5, None, :] == original_anomalies[None, :, :], axis=2), axis=1)
print(f"  Any of first 5 synthetic samples exact copies? {is_copy.any()} (should be False)")

In [ ]:
# Visualize: SMOTE creates new points in the minority class neighborhood
# Use PCA to project to 2D for visualization
pca_viz = PCA(n_components=2, random_state=RANDOM_STATE)

# Fit PCA on the original training data
X_train_pca = pca_viz.fit_transform(X_train_scaled)
X_smote_pca = pca_viz.transform(X_train_smote)

# Separate original and synthetic anomalies in the SMOTE-augmented set
n_original_anomalies = Counter(y_train)[1]
smote_anomaly_indices = np.where(y_train_smote == 1)[0]
original_anomaly_pca_idx = smote_anomaly_indices[:n_original_anomalies]
synthetic_anomaly_pca_idx = smote_anomaly_indices[n_original_anomalies:]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Left: Original data (before SMOTE) ---
axes[0].scatter(
    X_train_pca[y_train == 0, 0], X_train_pca[y_train == 0, 1],
    c='#2196F3', alpha=0.3, s=20, label=f'Normal ({Counter(y_train)[0]})'
)
axes[0].scatter(
    X_train_pca[y_train == 1, 0], X_train_pca[y_train == 1, 1],
    c='#F44336', alpha=0.9, s=60, marker='*',
    label=f'Anomaly ({Counter(y_train)[1]}) — original only'
)
axes[0].set_title('Before SMOTE (2D PCA Projection)\nFew anomalies, hard for model to learn boundary',
                  fontweight='bold', fontsize=11)
axes[0].set_xlabel(f'PC1 ({pca_viz.explained_variance_ratio_[0]*100:.1f}% var)')
axes[0].set_ylabel(f'PC2 ({pca_viz.explained_variance_ratio_[1]*100:.1f}% var)')
axes[0].legend(fontsize=9)

# --- Right: After SMOTE ---
axes[1].scatter(
    X_smote_pca[y_train_smote == 0, 0], X_smote_pca[y_train_smote == 0, 1],
    c='#2196F3', alpha=0.2, s=20, label=f'Normal ({Counter(y_train_smote)[0]})'
)
axes[1].scatter(
    X_smote_pca[original_anomaly_pca_idx, 0], X_smote_pca[original_anomaly_pca_idx, 1],
    c='#F44336', alpha=1.0, s=80, marker='*',
    label=f'Original anomalies ({n_original_anomalies})'
)
axes[1].scatter(
    X_smote_pca[synthetic_anomaly_pca_idx, 0], X_smote_pca[synthetic_anomaly_pca_idx, 1],
    c='#FF9800', alpha=0.5, s=30, marker='D',
    label=f'SMOTE synthetic ({len(synthetic_anomaly_pca_idx)})'
)
axes[1].set_title('After SMOTE (2D PCA Projection)\nSynthetic anomalies fill in the minority region',
                  fontweight='bold', fontsize=11)
axes[1].set_xlabel(f'PC1 ({pca_viz.explained_variance_ratio_[0]*100:.1f}% var)')
axes[1].set_ylabel(f'PC2 ({pca_viz.explained_variance_ratio_[1]*100:.1f}% var)')
axes[1].legend(fontsize=9)

plt.suptitle('SMOTE: Synthetic Minority Samples Fill the Anomaly Region',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Train and evaluate on SMOTE-augmented data
lr_smote = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = lr_smote.predict(X_test_scaled)

print("\nClassification Report (SMOTE):")
print(classification_report(y_test, y_pred_smote, target_names=['Normal (0)', 'Anomaly (1)']))

---

## Step 6: Technique 4 — Random Undersampling

Instead of adding more minority samples, we **reduce the majority class** — randomly removing normal samples until the classes are balanced.

**Trade-offs:**

| Aspect | Impact |
|---|---|
| Dataset size | Shrinks dramatically |
| Training time | Faster (less data) |
| Risk | Discards potentially useful information |
| When to use | Only when you have abundant data (millions of rows) |

**In our Nutanix scenario:** We have 2000 samples, ~1900 normal. After undersampling to balance classes, we'd have only ~200 total training samples — likely too few for a reliable model. In a real Nutanix deployment collecting millions of telemetry events per day, undersampling is more viable.

> **Instructor Note:** Point out the scale dependency. Undersampling makes sense for high-volume data streams (e.g., IPMI sensor data, Prism Central metrics). For small datasets (like a single cluster's worth of labeled incidents), undersampling throws away the little data you have. Ask: _"How much data does your team typically have for labeled failure events?"_

In [ ]:
# ----------------------------------------------------------------
# Random Undersampling — reduce majority class
# ----------------------------------------------------------------
rus = RandomUnderSampler(
    sampling_strategy='majority',  # only downsample the majority class
    random_state=RANDOM_STATE
)
X_train_rus, y_train_rus = rus.fit_resample(X_train_scaled, y_train)

print("=" * 55)
print("RANDOM UNDERSAMPLING")
print("=" * 55)
print(f"\nBefore undersampling: {Counter(y_train)}")
print(f"After undersampling : {Counter(y_train_rus)}")
print(f"\nDataset shrank from {len(y_train)} to {len(y_train_rus)} samples")
print(f"Discarded {len(y_train) - len(y_train_rus)} normal samples")
print(f"  ({(len(y_train)-len(y_train_rus))/len(y_train)*100:.1f}% of training data discarded!)")

# Visualize class distribution change
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

datasets = [
    ('Original', Counter(y_train)),
    ('After ROS', Counter(y_train_ros)),
    ('After RUS', Counter(y_train_rus))
]

for ax, (title, counts) in zip(axes, datasets):
    bars = ax.bar(
        ['Normal', 'Anomaly'],
        [counts[0], counts[1]],
        color=['#2196F3', '#F44336'],
        edgecolor='black',
        linewidth=0.8
    )
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_ylabel('Sample Count')
    ax.set_ylim(0, max(Counter(y_train_ros).values()) * 1.15)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
                f'{int(bar.get_height())}', ha='center', va='bottom', fontweight='bold')

plt.suptitle('Class Distribution Across Resampling Strategies', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Train and evaluate
lr_rus = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_rus.fit(X_train_rus, y_train_rus)
y_pred_rus = lr_rus.predict(X_test_scaled)

print("\nClassification Report (Random Undersampling):")
print(classification_report(y_test, y_pred_rus, target_names=['Normal (0)', 'Anomaly (1)']))

print("\nNote: Accuracy may be lower because we trained on much less data,")
print("but recall for anomalies may improve. Watch the precision-recall tradeoff.")

---

## Step 7: Choosing the Right Evaluation Metrics

This is arguably the most important section. **Choosing the wrong metric is how organizations deploy broken models and feel confident about it.**

### Why Accuracy Is Wrong for Imbalanced Data

With 5% anomaly rate:
- A model predicting all "Normal" gets **95% accuracy**
- But it has **0% recall** for failures — it catches nothing

### The Right Metrics

| Metric | Formula | What it answers |
|---|---|---|
| **Recall** (Sensitivity) | TP / (TP + FN) | "Of all actual failures, how many did we catch?" |
| **Precision** | TP / (TP + FP) | "Of all failure alerts, how many were real failures?" |
| **F1 Score** | 2 × P × R / (P + R) | Harmonic mean — balances precision and recall |
| **PR-AUC** | Area under PR curve | Overall model quality regardless of threshold |
| **ROC-AUC** | Area under ROC curve | Can be misleading with severe imbalance |

### PR-AUC vs. ROC-AUC: Which to Use?

**ROC-AUC** uses True Negative Rate — but with thousands of normal samples, even a bad model gets good TN rates. It **overestimates** model quality on imbalanced data.

**PR-AUC** focuses only on the positive (minority) class. It doesn't care how well you classify normal samples — it only rewards you for correctly finding failures. **Always prefer PR-AUC for anomaly detection.**

> **Instructor Note:** Ask ops engineers: _"Which would you rather have: a model with high ROC-AUC that misses 60% of failures, or a model with lower ROC-AUC but catches 90% of failures?"_ The answer is obvious when framed operationally. PR-AUC aligns with that intuition.

In [ ]:
# ----------------------------------------------------------------
# Precision-Recall Curves for all four techniques
# ----------------------------------------------------------------

# Collect models and their prediction probabilities
models_info = [
    ('Naive (no adjustment)', lr_naive,    y_pred_naive,    'gray',     '--'),
    ('Class Weights',         lr_weighted, y_pred_weighted, '#9C27B0',  '-'),
    ('Random Oversampling',   lr_ros,      y_pred_ros,      '#2196F3',  '-'),
    ('SMOTE',                 lr_smote,    y_pred_smote,    '#4CAF50',  '-'),
    ('Random Undersampling',  lr_rus,      y_pred_rus,      '#FF9800',  '-'),
]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# PR Curve
for name, model, y_pred, color, linestyle in models_info:
    # Get probabilities for the positive class
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    axes[0].plot(recall_vals, precision_vals,
                 label=f'{name}\n(PR-AUC={pr_auc:.3f})',
                 color=color, linestyle=linestyle, linewidth=2)

# Baseline: random classifier
baseline = y_test.mean()
axes[0].axhline(y=baseline, color='black', linestyle=':', linewidth=1.5,
                label=f'Random classifier (={baseline:.3f})')

axes[0].set_xlabel('Recall (Anomaly Detection Rate)', fontsize=12)
axes[0].set_ylabel('Precision (Accuracy of Alerts)', fontsize=12)
axes[0].set_title('Precision-Recall Curve\n(Higher = Better; Use PR-AUC for imbalanced data)',
                  fontweight='bold', fontsize=12)
axes[0].legend(loc='upper right', fontsize=8)
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.05])
axes[0].grid(True, alpha=0.3)

# ROC Curve for comparison
from sklearn.metrics import roc_curve
for name, model, y_pred, color, linestyle in models_info:
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = roc_auc_score(y_test, y_prob)
    axes[1].plot(fpr, tpr,
                 label=f'{name}\n(ROC-AUC={roc_auc:.3f})',
                 color=color, linestyle=linestyle, linewidth=2)

axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random classifier')
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate (Recall)', fontsize=12)
axes[1].set_title('ROC Curve\n(Note: Can be misleading with imbalanced data)',
                  fontweight='bold', fontsize=12)
axes[1].legend(loc='lower right', fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Evaluation Curves: PR (recommended) vs. ROC for Imbalanced Classification',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------------------------------------------
# Comprehensive comparison table: all 4 techniques
# ----------------------------------------------------------------

print("\n" + "=" * 85)
print("TECHNIQUE COMPARISON TABLE — Logistic Regression on Nutanix Anomaly Detection")
print("=" * 85)

results = []
for name, model, y_pred, color, ls in models_info:
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    results.append({
        'Technique': name,
        'Accuracy':  round((y_pred == y_test).mean(), 3),
        'Recall':    round(recall_score(y_test, y_pred), 3),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 3),
        'F1':        round(f1_score(y_test, y_pred, zero_division=0), 3),
        'ROC-AUC':   round(roc_auc_score(y_test, y_prob), 3),
        'PR-AUC':    round(average_precision_score(y_test, y_prob), 3),
    })

results_df = pd.DataFrame(results).set_index('Technique')

# Print formatted table
print(results_df.to_string())

print("\n" + "-" * 85)
print("KEY OBSERVATIONS:")
print("-" * 85)
print("  1. Naive approach has the HIGHEST accuracy — but lowest recall. It's useless.")
print("  2. All balancing techniques dramatically improve Recall and F1.")
print("  3. PR-AUC reveals differences that ROC-AUC masks.")
print("  4. SMOTE and Class Weights typically offer the best precision-recall balance.")
print("  5. Undersampling may sacrifice precision due to smaller training set.")

print("\nFor infrastructure monitoring, prioritize RECALL over Precision.")
print("  - Missing a failure = potential outage = high business cost")
print("  - False alert = ops team investigates unnecessarily = low cost")

# Highlight the best technique by recall
best_recall_idx = results_df['Recall'].idxmax()
best_pr_auc_idx  = results_df['PR-AUC'].idxmax()
print(f"\n  Best Recall : {best_recall_idx} ({results_df.loc[best_recall_idx, 'Recall']})")
print(f"  Best PR-AUC : {best_pr_auc_idx} ({results_df.loc[best_pr_auc_idx, 'PR-AUC']})")

---

## Step 8: Combining Techniques — The Production Recommendation

In practice, the best results come from **combining multiple techniques**:

1. **SMOTE** to generate diverse synthetic anomaly examples during training
2. **Class weights** on the model itself as a second layer of emphasis
3. **Random Forest** instead of Logistic Regression — better at capturing non-linear failure patterns

### Why Random Forest for Anomaly Detection?

- Handles non-linear interactions (e.g., high CPU *combined with* high latency is worse than either alone)
- Built-in feature importance tells you which metrics matter most
- Robust to outliers and noisy sensor data
- The `class_weight='balanced_subsample'` option applies weights per tree bootstrap — more stable than global balancing

> **Instructor Note:** This is the "production recipe." Walk through each component and why it's included. Key warning: **NEVER apply SMOTE to your test set**. The test set must reflect the real-world distribution to give you honest performance estimates. SMOTE on the test set would be data leakage. Use `imblearn.pipeline.Pipeline` in production to ensure this happens automatically.

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline

# ----------------------------------------------------------------
# Production-grade pipeline: SMOTE + Random Forest
# ----------------------------------------------------------------
# Using imblearn's Pipeline, which correctly handles resampling
# only during training (not during prediction/evaluation)
production_pipeline = ImbPipeline(steps=[
    ('smote', SMOTE(k_neighbors=5, random_state=RANDOM_STATE)),
    ('rf', RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced_subsample',   # apply weights at bootstrap level
        max_depth=15,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

# IMPORTANT: fit on original (unscaled) training data
# (Random Forests don't require feature scaling)
production_pipeline.fit(X_train, y_train)

# Evaluate on the held-out test set (no resampling applied here)
y_pred_prod = production_pipeline.predict(X_test)
y_prob_prod = production_pipeline.predict_proba(X_test)[:, 1]

print("=" * 65)
print("PRODUCTION PIPELINE: SMOTE + Random Forest (balanced_subsample)")
print("=" * 65)
print(classification_report(y_test, y_pred_prod,
                             target_names=['Normal (0)', 'Anomaly (1)']))

# Performance comparison
prod_metrics = {
    'Accuracy':  round((y_pred_prod == y_test).mean(), 3),
    'Recall':    round(recall_score(y_test, y_pred_prod), 3),
    'Precision': round(precision_score(y_test, y_pred_prod), 3),
    'F1':        round(f1_score(y_test, y_pred_prod), 3),
    'ROC-AUC':   round(roc_auc_score(y_test, y_prob_prod), 3),
    'PR-AUC':    round(average_precision_score(y_test, y_prob_prod), 3),
}
print("\nProduction Pipeline Metrics:")
for metric, value in prod_metrics.items():
    print(f"  {metric:<20}: {value}")

In [ ]:
# Feature importance — which metrics matter most for failure prediction?
rf_model = production_pipeline.named_steps['rf']
feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Feature importance
colors = ['#F44336' if imp > 0.15 else '#2196F3' for imp in feature_importance.values]
feature_importance.plot(
    kind='barh', ax=axes[0],
    color=colors[::-1], edgecolor='black', linewidth=0.5
)
axes[0].set_title('Feature Importance: What Predicts Failures?\n(Red = high importance)',
                  fontweight='bold', fontsize=12)
axes[0].set_xlabel('Feature Importance (Gini)')
axes[0].axvline(x=1/len(feature_cols), color='gray', linestyle='--', alpha=0.7,
                label=f'Equal importance ({1/len(feature_cols):.3f})')
axes[0].legend(fontsize=9)

# Confusion matrix for production model
cm_prod = confusion_matrix(y_test, y_pred_prod)
ConfusionMatrixDisplay(
    confusion_matrix=cm_prod,
    display_labels=['Normal', 'Anomaly']
).plot(ax=axes[1], colorbar=False, cmap='Greens')
axes[1].set_title(
    f'Production Model — Confusion Matrix\n'
    f'Recall: {recall_score(y_test, y_pred_prod):.2f} | '
    f'Precision: {precision_score(y_test, y_pred_prod):.2f} | '
    f'F1: {f1_score(y_test, y_pred_prod):.2f}',
    fontweight='bold', fontsize=11
)

plt.suptitle('Production SMOTE + Random Forest Model: Feature Importance & Performance',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop features for failure prediction:")
top3 = feature_importance.sort_values(ascending=False).head(3)
for feat, imp in top3.items():
    print(f"  {feat:<30}: {imp:.4f} ({imp/feature_importance.sum()*100:.1f}% of total)")
print("\nThis makes sense operationally — response time and error rate")
print("are the most direct indicators of user-impacting failures.")

---

## Step 9: Visualization — Decision Boundaries

Let's visualize how SMOTE changes the **decision boundary** learned by the model. We'll project to 2D using PCA and show the classification regions before and after SMOTE.

This visualization answers: _"Does SMOTE actually help the model learn a better boundary, or does it just move numbers around?"_

> **Instructor Note:** The decision boundary plots require some computational time. While running, explain that PCA is only for visualization — in production, the model uses all 8 features. The 2D projection loses some information, which is why the boundary looks simpler than it really is.

In [ ]:
from matplotlib.colors import ListedColormap

# Project training data to 2D via PCA for boundary visualization
pca_boundary = PCA(n_components=2, random_state=RANDOM_STATE)
pca_boundary.fit(X_train_scaled)

X_train_2d       = pca_boundary.transform(X_train_scaled)
X_train_smote_2d = pca_boundary.transform(X_train_smote)
X_test_2d        = pca_boundary.transform(X_test_scaled)

def plot_decision_boundary(ax, clf, X_2d, y, title, show_synthetic=False,
                            X_original_2d=None, y_original=None):
    """Train a classifier in 2D space and plot the decision boundary."""
    # Fit a LR in 2D space (for visualization only)
    lr_2d = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)
    lr_2d.fit(X_2d, y)

    # Create mesh grid
    h = 0.08
    x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
    y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    Z = lr_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    # Color map
    cmap_light = ListedColormap(['#BBDEFB', '#FFCDD2'])
    ax.contourf(xx, yy, Z, alpha=0.4, cmap=cmap_light)
    ax.contour(xx, yy, Z, colors=['#0D47A1'], linewidths=1.5, alpha=0.8)

    # Plot normal samples
    if show_synthetic and X_original_2d is not None:
        # Separate original anomalies from synthetic
        n_orig_anomalies = Counter(y_original)[1]
        all_anomaly_idx  = np.where(y == 1)[0]
        orig_anom_idx    = all_anomaly_idx[:n_orig_anomalies]
        synth_anom_idx   = all_anomaly_idx[n_orig_anomalies:]

        ax.scatter(X_2d[y == 0, 0], X_2d[y == 0, 1],
                   c='#2196F3', s=10, alpha=0.3, label='Normal')
        ax.scatter(X_2d[orig_anom_idx, 0], X_2d[orig_anom_idx, 1],
                   c='#F44336', s=80, marker='*', alpha=1.0, zorder=5, label='Original Anomaly')
        ax.scatter(X_2d[synth_anom_idx, 0], X_2d[synth_anom_idx, 1],
                   c='#FF9800', s=25, marker='D', alpha=0.5, label='SMOTE Synthetic')
    else:
        ax.scatter(X_2d[y == 0, 0], X_2d[y == 0, 1],
                   c='#2196F3', s=10, alpha=0.3, label='Normal')
        ax.scatter(X_2d[y == 1, 0], X_2d[y == 1, 1],
                   c='#F44336', s=80, marker='*', alpha=1.0, zorder=5, label='Anomaly')

    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.legend(fontsize=8, loc='upper right')

    return lr_2d

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Before SMOTE
plot_decision_boundary(
    axes[0],
    None, X_train_2d, y_train,
    'Before SMOTE: Decision Boundary\n(Few anomalies → uncertain boundary)'
)

# After SMOTE
plot_decision_boundary(
    axes[1],
    None, X_train_smote_2d, y_train_smote,
    'After SMOTE: Decision Boundary\n(Synthetic samples → clearer, more confident boundary)',
    show_synthetic=True,
    X_original_2d=X_train_2d,
    y_original=y_train
)

plt.suptitle('Decision Boundary: How SMOTE Changes What the Model Learns\n'
             '(Blue region = predicted Normal, Red region = predicted Anomaly)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nObservation:")
print("  Before SMOTE: The anomaly region is small and uncertain — the model")
print("  doesn't have enough anomaly examples to learn a confident boundary.")
print("  After SMOTE: Synthetic samples fill the anomaly region, helping the")
print("  model draw a clearer boundary between normal and failure states.")

### Final Comparison: All Techniques Side by Side

In [ ]:
# ----------------------------------------------------------------
# Final bar chart: head-to-head comparison of all techniques
# ----------------------------------------------------------------

# Add production model to results
all_results = results.copy()
all_results.append({
    'Technique': 'SMOTE + Random Forest\n(Production)',
    'Accuracy':  prod_metrics['Accuracy'],
    'Recall':    prod_metrics['Recall'],
    'Precision': prod_metrics['Precision'],
    'F1':        prod_metrics['F1'],
    'ROC-AUC':   prod_metrics['ROC-AUC'],
    'PR-AUC':    prod_metrics['PR-AUC'],
})

final_df = pd.DataFrame(all_results)

metrics_to_plot = ['Recall', 'Precision', 'F1', 'PR-AUC']
fig, axes = plt.subplots(1, 4, figsize=(20, 6))

palette = ['#9E9E9E', '#9C27B0', '#2196F3', '#4CAF50', '#FF9800', '#E91E63']

for ax, metric in zip(axes, metrics_to_plot):
    bars = ax.barh(
        range(len(final_df)),
        final_df[metric],
        color=palette,
        edgecolor='black',
        linewidth=0.6
    )
    ax.set_yticks(range(len(final_df)))
    ax.set_yticklabels(
        [t.replace('\n', ' ') for t in final_df['Technique']],
        fontsize=9
    )
    ax.set_title(metric, fontweight='bold', fontsize=13)
    ax.set_xlim(0, 1.05)
    ax.axvline(x=0.7, color='green', linestyle='--', alpha=0.5, linewidth=1)

    # Value labels on bars
    for bar, val in zip(bars, final_df[metric]):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', ha='left', fontsize=8)

    if metric == 'Recall':
        ax.set_xlabel('Score (1.0 = perfect)')

plt.suptitle('Technique Comparison: Anomaly Class Metrics\n'
             '(Green dashed line = 0.7 target threshold for production)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nFinal Rankings (by PR-AUC):")
final_df_sorted = final_df.sort_values('PR-AUC', ascending=False)
for rank, (_, row) in enumerate(final_df_sorted.iterrows(), 1):
    print(f"  {rank}. {row['Technique'].replace(chr(10), ' '):<45}: PR-AUC={row['PR-AUC']:.3f}, Recall={row['Recall']:.3f}, F1={row['F1']:.3f}")

---

## Checkpoint — Discussion Questions

Take 10 minutes to discuss these questions with your group. Apply the concepts to your actual Nutanix environment.

---

### Question 1: The Cost Asymmetry

In infrastructure monitoring, missing a failure (False Negative) is typically far more costly than a false alarm (False Positive). However, **too many false alarms** lead to "alert fatigue" — ops teams start ignoring alerts.

- What is an acceptable **false alarm rate** for your team's operations center?
- How does this translate to a target **Precision** value?
- How would you use the Precision-Recall curve to select a classification threshold that meets both constraints?

---

### Question 2: Which Technique Would You Deploy?

You are building an anomaly detection model for a Nutanix cluster with:
- **Scenario A:** 10 million telemetry events/day, 3-month history, 150 labeled failure incidents
- **Scenario B:** 50,000 telemetry events/day, 2-week history, 12 labeled failure incidents

Which resampling technique(s) would you recommend for each scenario, and why?

> _Hint: Think about dataset size, number of minority samples available for SMOTE, and the risk of overfitting._

---

### Question 3: SMOTE Limitations

SMOTE interpolates between existing minority samples. Consider these scenarios:

- **Failure type A:** CPU saturation from a rogue VM (smooth gradient in feature space)
- **Failure type B:** Sudden disk controller hardware fault (abrupt, point event)

For which failure type is SMOTE more appropriate? What alternative would you use for the other?

> _Hint: Consider whether failures form "clusters" in feature space or are scattered outliers._

---

### Question 4: Production Deployment

You've trained a great SMOTE + Random Forest model. You want to deploy it to Prism Central for real-time alerting. Identify at least **3 things that could go wrong** in production that weren't a concern during training in this notebook.

> _Consider: data drift, feature availability, resampling leakage, class distribution shifts over time, model staleness._

---

> **Instructor Note:** For Question 4, guide the discussion toward: (1) **Training-serving skew** — features computed differently in real-time vs. batch; (2) **Distribution drift** — infrastructure changes make historical anomalies unrepresentative; (3) **Threshold staleness** — the optimal threshold from training may not hold as patterns evolve; (4) **Label lag** — anomalies may not be labeled until hours after they occur; (5) **Pipeline leakage** — SMOTE accidentally applied during inference.

---

## Bonus: Threshold Tuning — Fine-Grained Control

By default, classifiers predict the positive class when `P(anomaly) > 0.5`. But you can **move this threshold** to trade off precision for recall.

For infrastructure monitoring: lowering the threshold catches more failures (higher recall) at the cost of more false alarms (lower precision). You choose the threshold based on your operational tolerance.

> **Instructor Note:** This is a quick but powerful concept. Real-world deployments almost always use a non-default threshold. In Nutanix's case, you might want different thresholds for different alert severities — high recall for page-level alerts, high precision for automated remediation actions.

In [ ]:
# Threshold analysis using the production model
y_prob_prod_test = production_pipeline.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.1, 0.91, 0.05)
threshold_results = []

for thresh in thresholds:
    y_pred_thresh = (y_prob_prod_test >= thresh).astype(int)
    if y_pred_thresh.sum() == 0:  # skip if no positive predictions
        continue
    threshold_results.append({
        'Threshold':    round(thresh, 2),
        'Recall':       round(recall_score(y_test, y_pred_thresh, zero_division=0), 3),
        'Precision':    round(precision_score(y_test, y_pred_thresh, zero_division=0), 3),
        'F1':           round(f1_score(y_test, y_pred_thresh, zero_division=0), 3),
        'Alerts/Day*':  int(y_pred_thresh.sum() / len(y_test) * 10000)  # scaled to 10k events/day
    })

thresh_df = pd.DataFrame(threshold_results)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Precision-Recall vs Threshold
axes[0].plot(thresh_df['Threshold'], thresh_df['Recall'],    'g-o', markersize=5, label='Recall',    linewidth=2)
axes[0].plot(thresh_df['Threshold'], thresh_df['Precision'], 'b-s', markersize=5, label='Precision', linewidth=2)
axes[0].plot(thresh_df['Threshold'], thresh_df['F1'],        'r-^', markersize=5, label='F1 Score',  linewidth=2)

# Mark the default threshold
axes[0].axvline(x=0.5, color='black', linestyle='--', alpha=0.7, label='Default threshold (0.5)')

# Find threshold with best F1
best_f1_row = thresh_df.loc[thresh_df['F1'].idxmax()]
axes[0].axvline(x=best_f1_row['Threshold'], color='purple', linestyle='--', alpha=0.7,
                label=f'Best F1 threshold ({best_f1_row["Threshold"]})')

axes[0].set_xlabel('Classification Threshold', fontsize=12)
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_title('Precision, Recall, F1 vs. Threshold\n(Trade off recall vs. precision by moving threshold)',
                  fontweight='bold', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Alert volume vs threshold
axes[1].plot(thresh_df['Threshold'], thresh_df['Alerts/Day*'], 'orange', marker='o',
             markersize=5, linewidth=2)
axes[1].axvline(x=best_f1_row['Threshold'], color='purple', linestyle='--', alpha=0.7)
axes[1].set_xlabel('Classification Threshold', fontsize=12)
axes[1].set_ylabel('Estimated Alerts per 10,000 Events/Day', fontsize=12)
axes[1].set_title('Alert Volume vs. Threshold\n(Lower threshold = more alerts = more ops burden)',
                  fontweight='bold', fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Threshold Tuning: Balancing Sensitivity vs. Alert Fatigue',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nThreshold Analysis Table:")
print(thresh_df.to_string(index=False))
print(f"\nRecommended threshold for this model: {best_f1_row['Threshold']} (maximizes F1)")
print(f"  At this threshold: Recall={best_f1_row['Recall']}, Precision={best_f1_row['Precision']}")
print(f"  Estimated alert volume: ~{best_f1_row['Alerts/Day*']} alerts per 10,000 events")

---

## Key Takeaways

### What We Learned

---

#### 1. Accuracy Is a Trap
With 5% anomaly rate, a model that predicts "normal" 100% of the time achieves 95% accuracy. **Never report accuracy alone for imbalanced classification**. Use Recall, F1, and PR-AUC.

---

#### 2. Four Techniques, Four Trade-Offs

| Technique | Best For | Watch Out For |
|---|---|---|
| **Class Weights** | First thing to try; zero cost | May not be enough for extreme imbalance |
| **Random Oversampling** | Quick improvement | Overfitting to repeated samples |
| **SMOTE** | Most datasets; general-purpose | Doesn't help if minority class is scattered noise |
| **Random Undersampling** | Very large datasets | Discards useful data; bad for small datasets |

---

#### 3. The Production Recipe

```
SMOTE (training data only)
+ class_weight='balanced_subsample'
+ Random Forest
+ Threshold tuning based on operational constraints
= Best results in production
```

**Critical rule:** Apply SMOTE **only to training data**. Never to test/validation data or in production inference.

---

#### 4. Use PR-AUC, Not ROC-AUC
ROC-AUC overestimates model quality with imbalanced data because it credits the model for correctly classifying the abundant majority class. PR-AUC focuses entirely on the minority class and is the honest metric for anomaly detection.

---

#### 5. Threshold Tuning Is an Operational Decision
The right threshold is not 0.5. It's determined by your operational constraints:
- How much alert fatigue can your team tolerate? → Sets minimum Precision
- What's the cost of a missed failure? → Sets minimum Recall
- Use the Precision-Recall curve to pick the operating point.

---

### In the Nutanix Context

These techniques apply directly to:
- **AHV host health scoring** — detecting compute node degradation before VM impact
- **Stargate storage anomalies** — catching I/O saturation before IOPS drop
- **CVM process health** — identifying memory leak patterns before OOM events
- **Network fabric monitoring** — flagging congestion patterns before packet loss

Every production monitoring ML system at scale deals with class imbalance. These are the tools to handle it correctly.

---

### Up Next: Lab 2.1
In the next module, we'll build **time-series anomaly detection** — combining the imbalance-handling techniques from this lab with temporal pattern recognition for sequential infrastructure data.

---

> **Instructor Note:** Summarize by asking: _"Given what you've learned today, how would you redesign the accuracy-based monitoring dashboards in your current environment?"_ Encourage engineers to think about their existing alert thresholds and whether they were set based on accuracy or on recall/precision. Often, existing thresholds are arbitrary — this lab gives them the vocabulary to demand better.

---
## 🎯 Your Turn — Challenges

These challenges extend the imbalanced dataset work from this lab.  
Use the `X_train`, `y_train`, `X_test`, `y_test` splits already defined above.

### Challenge 1 — SMOTE k-Neighbours Sensitivity

In Step 5 we used SMOTE with default `k_neighbors=5`.

**Task:**  
Run SMOTE with `k_neighbors=3` and `k_neighbors=7`, train a `LogisticRegression` on each, and compare:
- PR AUC on the test set
- Recall for the anomaly class
- Number of synthetic samples generated

Which k gives the best anomaly recall? Does higher k always mean better?

*Hint: `SMOTE(k_neighbors=3, random_state=42)`, `average_precision_score()`*

In [ ]:
# Challenge 1 — Your solution here




### Challenge 2 — False Alarm Cost Analysis

In real ops, a **false positive** (alerting on a healthy server) wastes an engineer's time.  
A **false negative** (missing a real failure) causes downtime.

**Task:**  
For each technique (Class Weights, SMOTE, Undersampling):
1. Calculate the **False Positive Rate** = FP / (FP + TN)
2. If the ops team reviews **500 alerts per day**, how many would be false alarms with each technique?
3. If a Nutanix cluster has **10,000 events/day** with 5% anomaly rate, how many real failures would each technique **miss**?

Build a summary DataFrame showing these business-impact metrics.

*Hint: Use `confusion_matrix()` to extract TN, FP, FN, TP values*

In [ ]:
# Challenge 2 — Your solution here




### Challenge 3 — Custom Decision Threshold

The production model uses a default 0.5 threshold. But for a Nutanix SRE team:
- Missing a failure (FN) costs 10x more than a false alarm (FP)

**Task:**  
Find the **optimal threshold** that minimises the weighted cost function:  
`cost = 10 * FN + 1 * FP`

Steps:
1. Get predicted probabilities from the production pipeline: `.predict_proba(X_test)[:, 1]`
2. Loop through thresholds from 0.1 to 0.9 (step 0.05)
3. For each threshold, calculate the weighted cost
4. Plot cost vs threshold and mark the minimum

What is the optimal threshold? How does it compare to 0.5?

In [ ]:
# Challenge 3 — Your solution here


